# Step 0: Pre-flight Check
This cell ensures a few things:
- The Python environment is configured
- `nebius` CLI is installed, with a valid profile
- `.env` is populated correctly

The cell below loads `.env`, checks that required entries are present, and reads the bucket with the provided keys.

**NO GPU jobs will be run**.

In [ ]:
from dotenv import load_dotenv
load_dotenv(".env")

import nebius_job as nj
import notebook_utils as nu

ready = nu.local_check()
print("\nReady to launch!" if ready else "\nfix the FAILs above (see docs/NEBIUS_SETUP.md)")

# Step 1: Serverless Job Launch (Dry Run)
The dry run prints what a launch would do:
1. **Build and push the job image**: NVIDIA's Isaac Sim 6.1 image with this repo's code copied in, tagged with a hash of the code. Both steps are quick when the code hasn't changed.
2. **The job command**, which runs that image with the configs injected and the shared filesystem mounted at `/shared` for Isaac's caches.

The code is in the image rather than injected because all of a job's injected files share a 64 KiB limit. See `docs/NEBIUS_SETUP.md`, step 7.

In [ ]:
nj.launch("configs/naive_20.json", dry_run=True)

# Step 2: Build the image and probe Nebius
`nj.launch()` builds and pushes the image itself; the cell below does it on its own so you can see it. The first push uploads the ~11 GB Isaac base layers once.

In [ ]:
tag = nj.build("configs/probe.json")

In [ ]:
probe_id = nj.launch("configs/probe.json")
probe = nj.wait(probe_id)
nj.logs(probe_id, tail=40)
print("\n", probe["state"], probe["durations_s"])

In [ ]:
nu.probe_frames(nj.last_job("probe")["name"])

# The Naive run

In [ ]:
naive_id = nj.launch("configs/naive_20.json")
naive_job = nj.wait(naive_id)
nj.logs(naive_id, tail=40)

In [ ]:
nu.fetch_run("configs/naive_20.json")
nu.summary("configs/naive_20.json")